In [1]:
import pandas as pd
import numpy as np
import requests
import json
from time import sleep

In [44]:
# --- Load files ---
pred_path = "data/tracking_data_play_2025090709.csv"   
# pred_path = "temp_909.csv"   
truth_path = "ben_truth.csv"  

pred = pd.read_csv(pred_path)
truth = pd.read_csv(truth_path)

key_cols = ["game_id", "gsis_play_id", "nfl_id"]
metric_cols = ["route_break_frame", "route_break_depth"]

# --- Force numeric dtypes, treat unparseable/missing as 0 ---
for col in metric_cols:
    pred[col] = pd.to_numeric(pred[col], errors="coerce").fillna(0)
    truth[col] = pd.to_numeric(truth[col], errors="coerce").fillna(0)

# --- Merge on identifying keys ---
merged = pred.merge(
    truth,
    on=key_cols,
    suffixes=("_pred", "_true"),
    how="inner"
)

print(f"Predicted rows: {len(pred)}, Ground truth rows: {len(truth)}, Matched rows: {len(merged)}")

unmatched_pred = pred.merge(truth[key_cols], on=key_cols, how="left", indicator=True)
missing_in_truth = unmatched_pred[unmatched_pred["_merge"] == "left_only"]
if len(missing_in_truth) > 0:
    print(f"Warning: {len(missing_in_truth)} predicted rows had no match in ground truth")

# --- Compute error metrics ---
def error_metrics(df, col, tolerance=None):
    diff = df[f"{col}_pred"] - df[f"{col}_true"]

    metrics = {
        "n": len(diff),
        "rmse": np.sqrt(np.mean(diff**2)),
        "mae": np.mean(np.abs(diff)),
        "bias": np.mean(diff),
        "std": np.std(diff),
        "max_abs_error": np.abs(diff).max(),
    }

    if tolerance is not None:
        within_tol = (np.abs(diff) <= tolerance).sum()
        tolerance_plus = tolerance + 1
        within_tol_plus = (np.abs(diff) <= tolerance_plus + 1).sum()
        metrics[f"pct_within_{tolerance}"] = within_tol / len(diff) * 100
        metrics[f"pct_within_{tolerance_plus}"] = within_tol_plus / len(diff) * 100

    return metrics

results = {
    "route_break_depth": error_metrics(merged, "route_break_depth", tolerance=1.0),
}

# --- Directional breakdown: is it over- or under-predicting? ---
def directional_breakdown(df, col, tolerance=None):
    diff = df[f"{col}_pred"] - df[f"{col}_true"]
    n = len(diff)

    over = (diff > 0).sum()
    under = (diff < 0).sum()
    exact = (diff == 0).sum()

    print(f"\n--- {col} directional breakdown ---")
    print(f"Overpredicted (pred > truth):  {over:4d}  ({over/n*100:5.1f}%)")
    print(f"Underpredicted (pred < truth): {under:4d}  ({under/n*100:5.1f}%)")
    print(f"Exact match:                   {exact:4d}  ({exact/n*100:5.1f}%)")

    if over > 0:
        print(f"  Avg overprediction magnitude:  {diff[diff > 0].mean():.3f}")
    if under > 0:
        print(f"  Avg underprediction magnitude: {diff[diff < 0].mean():.3f}")

    # If there's a tolerance, break down the misses specifically
    if tolerance is not None:
        misses = diff[np.abs(diff) > tolerance]
        if len(misses) > 0:
            miss_over = (misses > 0).sum()
            miss_under = (misses < 0).sum()
            print(f"  Of the {len(misses)} misses beyond ±{tolerance}: "
                  f"{miss_over} were overpredictions, {miss_under} were underpredictions")

    return {
        "pct_over": over / n * 100,
        "pct_under": under / n * 100,
        "avg_over_magnitude": diff[diff > 0].mean() if over > 0 else np.nan,
        "avg_under_magnitude": diff[diff < 0].mean() if under > 0 else np.nan,
    }

# --- Error by route type ---
def error_by_route(df, col, tolerance=None):
    route_col = "route_pred" if "route_pred" in df.columns else "route"
    rows = []
    for route, group in df.groupby(route_col):
        diff = group[f"{col}_pred"] - group[f"{col}_true"]
        row = {
            "route": route,
            "n": len(diff),
            "rmse": np.sqrt(np.mean(diff**2)),
            "mae": np.mean(np.abs(diff)),
            "bias": np.mean(diff),
        }
        if tolerance is not None:
            row[f"pct_within_{tolerance}"] = (np.abs(diff) <= tolerance).sum() / len(diff) * 100
        rows.append(row)
    return pd.DataFrame(rows).sort_values("mae", ascending=False)

results_df = pd.DataFrame(results).T
print("\nError metrics:")
print(results_df.round(3))

# --- Directional breakdown per route type ---
def directional_breakdown_by_route(df, col, tolerance=None):
    route_col = "route_pred" if "route_pred" in df.columns else "route"
    rows = []

    for route, group in df.groupby(route_col):
        diff = group[f"{col}_pred"] - group[f"{col}_true"]
        n = len(diff)

        over = (diff > 0).sum()
        under = (diff < 0).sum()
        exact = (diff == 0).sum()

        row = {
            "route": route,
            "n": n,
            "pct_over": over / n * 100,
            "pct_under": under / n * 100,
            "pct_exact": exact / n * 100,
            "avg_over_magnitude": diff[diff > 0].mean() if over > 0 else np.nan,
            "avg_under_magnitude": diff[diff < 0].mean() if under > 0 else np.nan,
        }

        if tolerance is not None:
            misses = diff[np.abs(diff) > tolerance]
            row["n_misses"] = len(misses)
            row["misses_over"] = (misses > 0).sum() if len(misses) > 0 else 0
            row["misses_under"] = (misses < 0).sum() if len(misses) > 0 else 0

        rows.append(row)

    return pd.DataFrame(rows).sort_values("n", ascending=False)

directional_breakdown(merged, "route_break_depth", tolerance=1.0)

route_error_df = error_by_route(merged, "route_break_depth", tolerance=1.0)
print("\nError by route type:")
print(route_error_df.round(3))

route_directional_df = directional_breakdown_by_route(merged, "route_break_depth", tolerance=1.0)
print("\nDirectional breakdown by route type:")
print(route_directional_df.round(2))

Predicted rows: 273, Ground truth rows: 273, Matched rows: 273

Error metrics:
                       n  rmse    mae  bias    std  max_abs_error  \
route_break_depth  273.0  2.37  1.128 -0.19  2.363           13.0   

                   pct_within_1.0  pct_within_2.0  
route_break_depth          82.051           90.11  

--- route_break_depth directional breakdown ---
Overpredicted (pred > truth):    68  ( 24.9%)
Underpredicted (pred < truth):   59  ( 21.6%)
Exact match:                    146  ( 53.5%)
  Avg overprediction magnitude:  1.882
  Avg underprediction magnitude: -3.051
  Of the 49 misses beyond ±1.0: 20 were overpredictions, 29 were underpredictions

Error by route type:
     route   n   rmse    mae   bias  pct_within_1.0
0    ANGLE  15  5.132  3.533 -2.200          53.333
1   CORNER  14  3.635  2.214  0.643          71.429
8     POST  18  3.145  2.111 -1.000          55.556
10   SLANT  12  2.677  1.333 -1.167          75.000
6       IN  22  2.153  1.182  0.091          81.

In [26]:
ppp = pd.read_csv("player_play_routes.csv")
ppp

C:\Users\benjamin.harris-mye\AppData\Local\Temp\ipykernel_15324\2906030672.py:1: DtypeWarning: Columns (0: drop_back_type, 1: drop_back_time, 2: pass_location_type, 3: shift_since_lineset, 4: pass_defended, 5: recovered_fumble, 6: caused_forced_fumble, 7: is_called_pass) have mixed types. Specify dtype option on import or set low_memory=False.
  ppp = pd.read_csv("player_play_routes.csv")


,game_id,gsis_play_id,nfl_id,season,season_type,week,game_key,team_id,opponent_team_id,team_abbr,...,fumble,fumble_lost,caused_forced_fumble,caused_pressure_at_pass_forward,is_called_pass,time_to_pressure,pass_blocking,missed_tackles_forced,motion_since_lineset,route
0,2017081252,1889,38703,2017,PRE,1,57181,1200,2510,DAL,...,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,OUT
1,2020102513,3908,46130,2020,REG,7,58283,2250,4400,JAX,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,False,OUT
2,2025102600,3229,57304,2025,REG,8,59952,2700,200,MIA,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,True,OUT
3,2019082963,1657,45607,2019,PRE,4,57896,3700,3430,PHI,...,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,GO
4,2025102600,3017,48489,2025,REG,8,59952,200,2700,ATL,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,False,GO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1107572,2021010307,3983,42369,2020,REG,17,58415,3430,3200,NYJ,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,False,SLANT
1107573,2019120104,317,42369,2019,REG,13,58083,4900,2250,TB,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,False,OUT
1107574,2022081851,4218,54694,2022,PRE,2,59127,4600,810,SEA,...,NaN,NaN,NaN,NaN,True,NaN,False,NaN,NaN,OUT
1107575,2019120805,2089,42489,2019,REG,14,58098,3000,1540,MIN,...,NaN,NaN,NaN,NaN,True,NaN,False,0.0,False,OUT


In [ ]:
compare_plays = pd.read_csv('tracking_data_play_2025121408.csv') 
compare_plays

,game_id,gsis_play_id,nfl_id,display_name,route_break_frame,route_break_depth
0,2.025121e+09,3401.0,54678.0,John FitzPatrick,2025121408-3401-246,6.0
1,2.025121e+09,3401.0,55907.0,Luke Musgrave,2025121408-3401-246,11.0
2,2.025121e+09,3401.0,56024.0,Dontayvion Wicks,2025121408-3401-237,4.0
3,2.025121e+09,3401.0,56516.0,Emanuel Wilson,NaN,NaN
4,2.025121e+09,3401.0,58225.0,Matthew Golden,2025121408-3401-246,14.0
...,...,...,...,...,...,...
278,2.025121e+09,3233.0,57223.0,Troy Franklin,NaN,NaN
279,2.025121e+09,4487.0,55907.0,Luke Musgrave,2025121408-4487-177,17.0
280,2.025121e+09,4487.0,56024.0,Dontayvion Wicks,2025121408-4487-192,31.0
281,2.025121e+09,4487.0,56375.0,Chris Brooks,2025121408-4487-193,2.0


In [ ]:
# Define parameters to access the identity API
USER = "nicholas.esposito"
API_KEY = "31bc0e9f40ca6fcf79f898de4c02b17deeb2be8cb91a26b7037198ef7bca00a2"

def pull_plays(game_id):

    url = f"https://api.profootballfocus.com/v1/a22/nfl/games/{game_id}/all_coverage/play"
    print(url)
    # Get JWT
    auth_resp = requests.post(
        "https://api.profootballfocus.com/auth/login",
        headers={"x-api-key": API_KEY}
    )
    print(auth_resp, jwt)

    # Use JWT for pass rush
    headers = {"Authorization": f"Bearer ${jwt}"}
    # print(headers)
    resp = requests.get(
        url,
        headers=headers,
        params={"page_size": 1}
    )

    res_json = resp.json()
    print(resp, 'asdfasdf')
    print(res_json)

    pff_route_depth = pd.json_normalize(res_json['route_depth'], sep = '-')

    # Filter to only what we need
    pff_route_depth_filtered = pff_route_depth

    # Access player API using JWT - we need nfl id from here (Note: gsis_id = NGS nfl_id)
    headers = {"Authorization": f"Bearer {jwt}"}
    resp = requests.get(
        url,
        headers=headers,
        params={"page_size": 1}
    )

    res_json = resp.json()
    print(res_json)

    pff_players = pd.json_normalize(res_json['players'], sep = '-')

    # Filter to only what we need
    pff_players_filtered = pff_players[['gsis_id', 'id']]

    # Now, join pff_players_filtered onto pff_route_depth_filtered (id = player_id) so the resulting dataframe has 'gsis_game_id', 'gsis_play_id', 'player_id', 'stance', and 'gsis_id'
    pff_merged = pff_route_depth_filtered.merge(
        pff_players_filtered,
        left_on = "player_id",
        right_on = "id",
        how = "left"
    )

    return pff_merged

In [ ]:
pull_plays('28635')

https://api.profootballfocus.com/v1/a22/nfl/games/28635/all_coverage/play
<Response [200]> eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhdWQiOiJQRkYuSG9kb3IiLCJleHAiOjE3ODg4MTQzMTYsImlhdCI6MTc4ODgxMDcxNiwiaXNzIjoiUEZGLkhvZG9yIiwianRpIjoiNjZmZDZjNWQtZGQ2NS00NDRhLTlkMGMtYjBjYzNhMGU4YTM5IiwibmJmIjoxNzg4ODEwNzE1LCJzdWIiOiJ7XCJhY2Nlc3NfdG9rZW5cIjpcIjlRcmJSRmRubVIzdWpzNFoyZHVLLUFsRUY3YzRVVmdyUm1Rcm5VejZSVGVVemlZVHVKZnhsZkI1aXVZV0p3M25cIixcImFwaV9rZXlcIjpcIjMxYmMwZTlmNDBjYTZmY2Y3OWY4OThkZTRjMDJiMTdkZWViMmJlOGNiOTFhMjZiNzAzNzE5OGVmN2JjYTAwYTJcIixcImV4cGlyZXNfaW5cIjozNjAwLFwidXNlclwiOntcImNvbnN0cmFpbnRzXCI6e1wiZnJhbmNoaXNlXCI6W1wiXCJdfSxcImVtYWlsXCI6XCJhcGlrZXlATkZMLmNvbVwiLFwiZmlyc3RfbmFtZVwiOlwiQVBJIEtleVwiLFwibGFzdF9uYW1lXCI6XCJOR1NcIixcInBlcm1pc3Npb25zXCI6e1wiYXBpXCI6W1wiYTIyXCIsXCJtYXN0ZXJfZGF0YVwiLFwicHJlbWl1bVwiLFwidmlkZW9cIl0sXCJsZWFndWVcIjpbXCJuY2FhXCIsXCJuZmxcIl19LFwidWlkXCI6XCJjNmVjZGM4MC1lMDVmLTRjOWQtODMzMi02MzNmNzE3ZDdkZWJcIixcInZlcnRpY2FsXCI6XCJCMkJcIn19IiwidHlwIjoiYWNjZXNzIn0.PyQjDy-x

KeyError: 'route_depth'